# 01 · Data preparation

**Goal.** Build a clean, fully encoded dataset from the original TCGA export and prove that it is identical to the encoded file published by UCI.

Why rebuild it instead of using the UCI file directly? The UCI file (`TCGA_InfoWithGrade.csv`) drops the patient identifier (`Case_ID`) and the histological diagnosis (`Primary_Diagnosis`). We need the identifier to trace patients and the diagnosis to (i) demonstrate the leakage problem and (ii) prepare the multiclass task planned for January. Rebuilding from `TCGA_GBM_LGG_Mutations_all.csv` and checking the result value by value against UCI gives both: the extra columns and the guarantee that our cleaning is exactly the published one.

**Outputs** (all in `results/`):
- `tables/data_dictionary.tex`, `tables/data_cleaning.tex`, `tables/data_uci_check.tex`, `tables/data_feature_sets.tex`
- `logs/01_data_preparation.json`
- `data/processed/glioma_clean.csv` (regenerated each run, not versioned)

In [1]:
# Make the project root importable, whatever folder the notebook is run from
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Project root (folder containing src/config.py) not found")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display

from src import config, data, reporting

pd.set_option("display.max_columns", 40)
print("Project root:", ROOT)

Project root: C:\Users\ori\ML


## 1. Load the raw files

Two files are provided. `Mutations_all` keeps the original text labels; `InfoWithGrade` is the version UCI encoded as integers after removing incomplete patients. The raw files are only read, never modified.

In [2]:
raw = data.load_raw_mutations()
uci = data.load_uci_encoded()

print(f"Original file : {raw.shape[0]} patients x {raw.shape[1]} columns")
print(f"UCI file      : {uci.shape[0]} patients x {uci.shape[1]} columns")
print("Columns only in the original file:", sorted(set(raw.columns) - set(uci.columns)))
display(raw.head())

Original file : 862 patients x 27 columns
UCI file      : 839 patients x 24 columns
Columns only in the original file: ['Case_ID', 'Primary_Diagnosis', 'Project']


,Grade,Project,Case_ID,Gender,Age_at_diagnosis,Primary_Diagnosis,Race,IDH1,TP53,ATRX,PTEN,EGFR,CIC,MUC16,PIK3CA,NF1,PIK3R1,FUBP1,RB1,NOTCH1,BCOR,CSMD3,SMARCA4,GRIN2A,IDH2,FAT4,PDGFRA
0,LGG,TCGA-LGG,TCGA-DU-8164,Male,51 years 108 days,"Oligodendroglioma, NOS",white,MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,MUTATED,NOT_MUTATED,NOT_MUTATED,MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED
1,LGG,TCGA-LGG,TCGA-QH-A6CY,Male,38 years 261 days,Mixed glioma,white,MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED
2,LGG,TCGA-LGG,TCGA-HW-A5KM,Male,35 years 62 days,"Astrocytoma, NOS",white,MUTATED,MUTATED,MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED
3,LGG,TCGA-LGG,TCGA-E1-A7YE,Female,32 years 283 days,"Astrocytoma, anaplastic",white,MUTATED,MUTATED,MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,MUTATED,NOT_MUTATED,NOT_MUTATED,MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,MUTATED,NOT_MUTATED
4,LGG,TCGA-LGG,TCGA-S9-A6WG,Male,31 years 187 days,"Astrocytoma, anaplastic",white,MUTATED,MUTATED,MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED,NOT_MUTATED


## 2. Data quality

Missing values are not stored as empty cells: the TCGA export writes `--`, and race can also be `not reported`. A plain `isna()` would find nothing, so we count these markers explicitly. We also check that no patient appears twice.

In [3]:
quality = data.count_invalid_markers(raw)
display(quality)
print("Duplicated Case_ID:", raw[config.ID_COL].duplicated().sum())
print("Values found in the gene columns:", sorted(set(raw[config.GENE_COLS].stack())))

,missing ('--'),'not reported',NaN
Gender,4,0,0
Age_at_diagnosis,5,0,0
Primary_Diagnosis,4,0,0
Race,4,18,0


Duplicated Case_ID: 0
Values found in the gene columns: ['MUTATED', 'NOT_MUTATED']


## 3. Leakage columns

`Project` (TCGA-LGG / TCGA-GBM) is the TCGA sub-project each patient comes from. If it coincides with `Grade`, it is the target under another name and must be dropped. `Primary_Diagnosis` is kept only as metadata: the cross-tabulation shows that the diagnosis determines the grade, so it can never be a predictor of `Grade` (and vice versa).

In [4]:
project_is_grade = data.check_leakage_columns(raw)
print("Project identical to Grade:", project_is_grade)
display(pd.crosstab(raw[config.MULTICLASS_TARGET], raw["Grade"], margins=True))

Project identical to Grade: True


Grade,GBM,LGG,All
Primary_Diagnosis,,,
--,3,1,4
"Astrocytoma, NOS",0,58,58
"Astrocytoma, anaplastic",0,129,129
Glioblastoma,360,0,360
Mixed glioma,0,128,128
"Oligodendroglioma, NOS",0,108,108
"Oligodendroglioma, anaplastic",0,75,75
All,363,499,862


## 4. Remove incomplete patients

A patient is removed if a clinical field needed by the project is missing. Each patient is counted under the first reason that applies, so the counts add up. The removed patients are listed below to check that they are not concentrated in one class.

In [5]:
clean, removal_table, removed = data.clean_and_encode(raw)
display(removal_table)
print("Grade of the removed patients:", removed["Grade"].value_counts().to_dict())
display(removed[[config.ID_COL, "Grade", "Gender", "Age_at_diagnosis", "Race", "Reason"]])

,Step,Removed,Remaining
0,Original file,0,862
1,All clinical fields missing,4,858
2,Race not reported,18,840
3,Age at diagnosis missing,1,839


Grade of the removed patients: {'LGG': 12, 'GBM': 11}


,Case_ID,Grade,Gender,Age_at_diagnosis,Race,Reason
41,TCGA-R8-A6YH,LGG,--,--,--,All clinical fields missing
71,TCGA-CS-6290,LGG,Male,31 years 344 days,not reported,Race not reported
163,TCGA-DB-A64Q,LGG,Female,31 years 315 days,not reported,Race not reported
208,TCGA-RY-A840,LGG,Male,47 years 335 days,not reported,Race not reported
231,TCGA-VM-A8CB,LGG,Male,33 years 287 days,not reported,Race not reported
256,TCGA-VV-A86M,LGG,Female,36 years 304 days,not reported,Race not reported
268,TCGA-DB-A75P,LGG,Female,25 years 282 days,not reported,Race not reported
341,TCGA-DB-5279,LGG,Male,59 years 359 days,not reported,Race not reported
396,TCGA-VM-A8C9,LGG,Female,37 years 149 days,not reported,Race not reported
437,TCGA-W9-A837,LGG,Male,--,white,Age at diagnosis missing


## 5. Encoding

Age is converted from `"X years Y days"` to years (`X + Y/365`, rounded to 2 decimals). All categorical variables are encoded as integers with the same codes as UCI (see the data dictionary below).

`Race` is **nominal**: the codes 0-3 have no order, so treating them as a number would be wrong for linear models and SVMs. It is kept with the UCI codes here and one-hot encoded inside each model pipeline (stages 4 and 5).

In [6]:
display(clean.head())
print(clean.dtypes.value_counts())
dictionary = data.data_dictionary()
display(dictionary)

,Case_ID,Primary_Diagnosis,Grade,Gender,Age_at_diagnosis,Race,IDH1,TP53,ATRX,PTEN,EGFR,CIC,MUC16,PIK3CA,NF1,PIK3R1,FUBP1,RB1,NOTCH1,BCOR,CSMD3,SMARCA4,GRIN2A,IDH2,FAT4,PDGFRA
0,TCGA-DU-8164,"Oligodendroglioma, NOS",0,0,51.30,0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0
1,TCGA-QH-A6CY,Mixed glioma,0,0,38.72,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,TCGA-HW-A5KM,"Astrocytoma, NOS",0,0,35.17,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,TCGA-E1-A7YE,"Astrocytoma, anaplastic",0,1,32.78,0,1,1,1,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0
4,TCGA-S9-A6WG,"Astrocytoma, anaplastic",0,0,31.51,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


int64      23
object      2
float64     1
Name: count, dtype: int64


,Variable,Type,Encoding,Role
0,Case_ID,Identifier,TCGA patient barcode,Identifier (not a predictor)
1,Primary_Diagnosis,Nominal (6 classes),Histological diagnosis (text),Excluded: determines Grade (leakage)
2,Project,Nominal (2 classes),TCGA-LGG / TCGA-GBM,Dropped: identical to Grade
3,Grade,Binary,"0 = LGG, 1 = GBM",Classification target
4,Age_at_diagnosis,Continuous,"Years (years + days/365, 2 decimals)",Regression target; predictor for classification
5,Gender,Binary,"0 = male, 1 = female",Predictor
6,Race,Nominal (4 classes),"0 = white, 1 = black or African American, 2 = ...",Predictor (one-hot encoded inside the model pi...
7,"20 genes (IDH1, TP53, ATRX, PTEN, EGFR, CIC, M...",Binary,"0 = not mutated, 1 = mutated",Predictors; input of the unsupervised analysis


## 6. Check against the UCI file

The rebuilt dataset must coincide with `TCGA_InfoWithGrade.csv` in the number of patients, their order and every value. If any check fails, the cleaning is not the published one and the notebook stops.

In [7]:
checks = data.verify_against_uci(clean, uci)
uci_check = pd.DataFrame({"Check": list(checks), "Passed": list(checks.values())})
display(uci_check)
assert all(checks.values()), "The rebuilt dataset differs from the UCI file"
print("The rebuilt dataset is identical to the UCI file.")

,Check,Passed
0,Same number of patients,True
1,Grade identical,True
2,Gender identical,True
3,Race identical,True
4,IDH1 identical,True
5,TP53 identical,True
6,ATRX identical,True
7,PTEN identical,True
8,EGFR identical,True
9,CIC identical,True


The rebuilt dataset is identical to the UCI file.


## 7. Final dataset and feature sets

Two facts to keep in mind for the next stages:
- **Rare categories.** Some race codes have very few patients (see counts). In cross-validation a category may be absent from a training fold; the one-hot encoder will be set to ignore unknown categories.
- **Sparse mutations.** Several genes are mutated in fewer than 5% of patients. They are kept (they may still be informative), but the EDA will quantify their individual relevance.

In [8]:
print("Grade:", clean["Grade"].map(config.CLASS_LABELS).value_counts().to_dict())
print("Race :", clean["Race"].value_counts().sort_index().to_dict())
prevalence = clean[config.GENE_COLS].mean().sort_values()
print("Genes mutated in < 5% of patients:", prevalence[prevalence < 0.05].round(3).to_dict())

feature_sets = pd.DataFrame([
    {"Task": "Classification", "Target": "Grade",
     "Predictors": ", ".join(data.feature_columns("classification")[:3]) + " + 20 genes",
     "Number of predictors": len(data.feature_columns("classification"))},
    {"Task": "Regression", "Target": "Age_at_diagnosis",
     "Predictors": ", ".join(data.feature_columns("regression")[:2]) + " + 20 genes",
     "Number of predictors": len(data.feature_columns("regression"))},
    {"Task": "Unsupervised", "Target": "None",
     "Predictors": "20 genes", "Number of predictors": len(data.feature_columns("unsupervised"))},
])
display(feature_sets)

Grade: {'LGG': 487, 'GBM': 352}
Race : {0: 765, 1: 59, 2: 14, 3: 1}
Genes mutated in < 5% of patients: {'PDGFRA': 0.026, 'IDH2': 0.027, 'FAT4': 0.027, 'SMARCA4': 0.032, 'GRIN2A': 0.032, 'CSMD3': 0.032, 'BCOR': 0.035, 'NOTCH1': 0.045, 'RB1': 0.048}


,Task,Target,Predictors,Number of predictors
0,Classification,Grade,"Gender, Age_at_diagnosis, Race + 20 genes",23
1,Regression,Age_at_diagnosis,"Gender, Race + 20 genes",22
2,Unsupervised,None,20 genes,20


## 8. Save results

In [9]:
processed_path = data.save_processed(clean)

reporting.save_table_tex(dictionary, "data_dictionary", column_format="p{3.2cm}p{2.4cm}p{4.6cm}p{4.2cm}")
reporting.save_table_tex(removal_table, "data_cleaning", column_format="lrr")
reporting.save_table_tex(uci_check, "data_uci_check", column_format="ll")
reporting.save_table_tex(feature_sets, "data_feature_sets", column_format="llp{6cm}r")

summary = {
    "n_raw": len(raw),
    "n_clean": len(clean),
    "n_removed": len(removed),
    "removal_steps": removal_table,
    "removed_by_grade": removed["Grade"].value_counts().to_dict(),
    "removed_case_ids": removed[config.ID_COL].tolist(),
    "duplicated_case_ids": int(raw[config.ID_COL].duplicated().sum()),
    "invalid_markers_per_column": quality.to_dict(orient="index"),
    "project_identical_to_grade": project_is_grade,
    "diagnosis_vs_grade": pd.crosstab(raw[config.MULTICLASS_TARGET], raw["Grade"]).to_dict(orient="index"),
    "uci_checks": checks,
    "identical_to_uci": all(checks.values()),
    "grade_counts": clean["Grade"].map(config.CLASS_LABELS).value_counts().to_dict(),
    "grade_gbm_proportion": float(clean["Grade"].mean()),
    "race_counts": clean["Race"].value_counts().sort_index().to_dict(),
    "gender_counts": clean["Gender"].value_counts().sort_index().to_dict(),
    "age_summary": clean["Age_at_diagnosis"].describe().round(2).to_dict(),
    "gene_prevalence": prevalence.round(4).to_dict(),
    "feature_sets": {t: data.feature_columns(t) for t in ("classification", "regression", "unsupervised")},
    "processed_file": str(processed_path.relative_to(config.ROOT)),
}
log_path = reporting.save_json(summary, "01_data_preparation")
print("Saved:", processed_path.relative_to(config.ROOT), "and", log_path.relative_to(config.ROOT))
print("Tables:", sorted(p.name for p in config.TABLES.glob("data_*.tex")))

Saved: data\processed\glioma_clean.csv and results\logs\01_data_preparation.json
Tables: ['data_cleaning.tex', 'data_dictionary.tex', 'data_feature_sets.tex', 'data_uci_check.tex']
